# w13a — Multiple Linear Regression in Python
**ENGR 1451/2451 · Exploratory Data Science · Fall 2026**

Fit models with `smf.ols`, use `C()` for categorical predictors, compare $R^2$ and adjusted $R^2$, and test a regional contrast.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

## 1. Read the book data and fit two models

`weight` (g) is the response, `volume` (cm³) is numerical, and `cover` is `hb` (hardback) or `pb` (paperback).

The formula `weight ~ volume + C(cover)` includes an intercept and uses **hardback as the reference**. Its paperback coefficient compares books of the *same volume*.

In [ ]:
allbacks = pd.read_csv("data/allbacks.csv")
display(allbacks.head())

In [ ]:
book_simple = smf.ols("weight ~ volume", data=allbacks).fit()
book_model = smf.ols("weight ~ volume + C(cover)", data=allbacks).fit()
print(book_model.summary().tables[1])

## 2. Compare $R^2$ and adjusted $R^2$

For nested OLS models fitted to the **same observations**, adding a predictor cannot reduce ordinary $R^2$. Adjusted $R^2$ accounts for the number of predictors, so it **can decrease**.

Add `area` (hard-board area, recorded as 0 for paperbacks). **Discuss:** Does adjusted $R^2$ favor adding `area`?

In [ ]:
book_area = smf.ols("weight ~ volume + C(cover) + area", data=allbacks).fit()
fits = {"Volume": book_simple, "Volume + cover": book_model,
        "Volume + cover + area": book_area}
display(pd.DataFrame({label: {"R²": fit.rsquared,
                              "Adjusted R²": fit.rsquared_adj}
                      for label, fit in fits.items()}).T.round(4))

## 3. Filter and rename county data

Each row is a **county**, not an individual. We keep the same 3,140 complete-case counties used in the slides. `.loc` selects rows and columns; `.rename()` changes a selected copy without renaming `county` itself.

In [ ]:
county_file = Path("data/county_complete.csv")
raw = pd.read_csv(county_source, dtype={"fips": "string"})

needed = ["state", "poverty_2017", "hs_grad_2017", "bachelors_2017"]
county = raw.dropna(subset=needed).copy()
print(f"Loaded {len(raw):,} rows; retained {len(county):,} complete counties")

display(county.loc[county["state"] == "Pennsylvania",
                   ["name", "poverty_2017"]]
        .rename(columns={"name": "county_name", "poverty_2017": "poverty_pct"})
        .head())

## 4. Choose a reference category and test a contrast

The county file has state names but no region column. Build `region` from states and specify its category order with `pd.Categorical`. With **Northeast as the first level**, each other region coefficient compares that region's mean county poverty rate with Northeast. These are **unweighted** county means.

In [ ]:
states_by_region = {
    "Northeast": ["Connecticut", "Maine", "Massachusetts", "New Hampshire", "Rhode Island",
                  "Vermont", "New Jersey", "New York", "Pennsylvania"],
    "Midwest": ["Illinois", "Indiana", "Michigan", "Ohio", "Wisconsin", "Iowa", "Kansas",
                "Minnesota", "Missouri", "Nebraska", "North Dakota", "South Dakota"],
    "West": ["Arizona", "Colorado", "Idaho", "Montana", "Nevada", "New Mexico", "Utah",
             "Wyoming", "Alaska", "California", "Hawaii", "Oregon", "Washington"],
    "South": ["Delaware", "District of Columbia", "Florida", "Georgia", "Maryland",
              "North Carolina", "South Carolina", "Virginia", "West Virginia", "Alabama",
              "Kentucky", "Mississippi", "Tennessee", "Arkansas", "Louisiana", "Oklahoma", "Texas"]
}
state_to_region = {state: region for region, states in states_by_region.items()
                   for state in states}
county["region"] = pd.Categorical(county["state"].map(state_to_region),
                                  categories=list(states_by_region))
assert county["region"].notna().all()  # check that every state was mapped

region_model = smf.ols("poverty_2017 ~ C(region)", data=county).fit()
display(county.groupby("region", observed=True)["poverty_2017"]
        .agg(n="size", mean="mean").round(3))
print(region_model.summary().tables[1])

### Test West against South directly

The West and South coefficient P-values separately compare each region with **Northeast**. To test whether West and South differ, test $H_0:\beta_W-\beta_S=0$ directly.

In [ ]:
# This contrasts West with South, not either one with Northeast:
print(region_model.t_test("C(region)[T.West] - C(region)[T.South] = 0"))

## 5. Check correlated income predictors and residuals

`M` is median household income and `P` is per-capita income, each measured in **thousands of dollars**. Fit both income models to the same counties, then compare the median-income slope, its standard error, and adjusted $R^2$.

**Discuss:** What happens to slope uncertainty when highly correlated predictors enter the same model? Look for curvature, changing spread, or unusual points in the residual plot. OLS inference does not automatically account for geographic dependence or uncertainty in estimated county rates.

In [ ]:
income = county.dropna(subset=["median_household_income_2017",
                               "per_capita_income_2017"]).copy()
income["M"] = income["median_household_income_2017"] / 1000
income["P"] = income["per_capita_income_2017"] / 1000
income_one = smf.ols("poverty_2017 ~ M", data=income).fit()
income_two = smf.ols("poverty_2017 ~ M + P", data=income).fit()

print(f"Predictor correlation r = {income['M'].corr(income['P']):.3f}")
display(pd.DataFrame({
    "Model": ["M only", "M + P"],
    "M slope": [income_one.params["M"], income_two.params["M"]],
    "M slope SE": [income_one.bse["M"], income_two.bse["M"]],
    "Adjusted R²": [income_one.rsquared_adj, income_two.rsquared_adj],
}).set_index("Model").round(4))

plt.figure(figsize=(6, 3.4))
plt.scatter(income_two.fittedvalues, income_two.resid, s=10, alpha=0.4)
plt.axhline(0, color="0.3", linestyle="--")
plt.xlabel("Fitted poverty rate (%)")
plt.ylabel("Residual (percentage points)")
plt.title("Income model: check for patterns")
plt.tight_layout()
plt.show()

## 6. Summary and Python function reference

**Interpret a coefficient relative to the other predictors held fixed.** Use adjusted $R^2$ to compare model fits with different numbers of predictors, and check residuals before trusting model-based inference.

| Function or attribute | Purpose |
|---|---|
| `pd.read_csv(path)` | Read a dataset. |
| `df.loc[rows, columns]` | Select rows and columns. |
| `df.rename(columns={...})` | Rename columns in a selected copy. |
| `df["state"].isin([...])` | Select observations belonging to specified groups. |
| `pd.Categorical(values, categories=[...])` | Set category order and a reference level. |
| `smf.ols("y ~ x + C(group)", data=df).fit()` | Fit numerical and categorical predictors. |
| `model.params`, `model.bse`, `model.pvalues` | Read coefficients, standard errors, and P-values. |
| `model.rsquared`, `model.rsquared_adj` | Compare $R^2$ and adjusted $R^2$. |
| `model.t_test("term1 - term2 = 0")` | Test a coefficient contrast. |
| `model.predict(new_data)` | Predict for specified predictor values. |
| `model.fittedvalues`, `model.resid` | Inspect fitted responses and residuals. |

## 7. Your turn — books and counties

1. Use `.loc` to show **paperbacks with volume at least 800 cm³**, keeping only `volume` and `weight`.
2. Use `.isin()` to show **Pennsylvania and Ohio** counties. Rename only the displayed `name` and `poverty_2017` columns to `county_name` and `poverty_pct`. Does this change `county`?
3. Change the region reference to **South**, refit `poverty_2017 ~ C(region_south)`, and compare its **West coefficient** with the original West-minus-South contrast. Does $R^2$ change?
4. Use `book_model.predict()` to estimate weights for a **600 cm³ hardcover and paperback**. What is the predicted weight difference?

**Explain:** Why can adding `area` lower adjusted $R^2$ without lowering $R^2$? Why might correlated income predictors increase an individual slope's standard error even if adjusted $R^2$ improves?

Use the code patterns above; no manual regression calculations are needed.

In [ ]:
# 1. Filter allbacks using .loc.

# 2. Select Pennsylvania and Ohio using .isin(), then rename the displayed copy.

# 3. Set South as the first category, refit, and compare with region_model.

# 4. Predict the weights of these two books and compare:
books_600 = pd.DataFrame({"volume": [600, 600], "cover": ["hb", "pb"]})
# Complete the prediction and difference calculation here.